# CodeSentinel — triage model

Trains the optional ranking model and exports the two files the CLI expects.

**Read this before quoting any number from this notebook.**

The model is not a detector. Findings come from deterministic rules; this model
only (a) orders findings within a severity band and (b) raises a *needs review*
hint when it scores a class highly and no rule fired. It never creates a finding
and never names a CWE. So a number from here is a **ranking** number, not a
detection rate, and it must never be presented as "CodeSentinel detects X% of
vulnerabilities".

Two things decide whether the numbers mean anything at all:

* **The split is by group, never by row.** A vulnerable function and its own
  fixed twin share a group. Splitting on rows puts them on opposite sides and
  every metric jumps — that is the most common way a vulnerability dataset lies.
* **The scaler is fitted on the training fold only.** Fitting on everything
  leaks the test distribution into the model's input range.

Both are enforced in `scripts/train_triage.py`, which this notebook calls. The
logic lives there rather than in cells so that what you run here is exactly what
was tested.

In [ ]:
# Kaggle: enable Internet in the sidebar (Settings -> Internet -> On).
!git clone -q https://github.com/AnishPrakash/codesentinel.git
%cd codesentinel
!pip install -q -r requirements.txt
!pip install -q -e .
!pip install -q torch pandas scikit-learn onnx onnxscript
print("ready")

## 1. Collect the sources

Three sources, in increasing order of how much the numbers are worth.

| Source | Size | Language | Labels | Worth |
|---|---|---|---|---|
| OWASP Benchmark | 2,740 cases | Java | CWE + true/false, by the project | Generated test cases. Good hard negatives, but a model that scores well has learned this generator's idioms. |
| Juliet (NIST) | ~28k cases | Java | CWE in the path, `bad()`/`good*()` methods | Same caveat — generated. |
| CVEfixes | ~5k methods | Python, JS, Java | CVE → CWE, before/after a real fix | **The only one made of code people shipped.** The pair is the point: same author, same style, one difference. |

Run all three if you can. OWASP alone is not enough — see section 5.

In [ ]:
# OWASP Benchmark — clones itself, no download needed
!python scripts/dataset/collect_owasp_benchmark.py --clone

In [ ]:
# Juliet — attach the NIST Java suite as a Kaggle dataset, or download it here.
# https://samate.nist.gov/SARD/test-suites/juliet
JULIET_ROOT = "/kaggle/input/juliet-java"      # <- change to your path
import os
if os.path.exists(JULIET_ROOT):
    !python scripts/dataset/collect_juliet.py --root {JULIET_ROOT} --limit 200
    print("--- smoke test above. If the counts look right, rerun without --limit ---")
    !python scripts/dataset/collect_juliet.py --root {JULIET_ROOT}
else:
    print(f"{JULIET_ROOT} not found - skipping Juliet")

In [ ]:
# CVEfixes — attach the SQLite dump as a Kaggle dataset.
# https://github.com/secureIT-project/CVEfixes
CVEFIXES_DB = "/kaggle/input/cvefixes/CVEfixes.db"   # <- change to your path
if os.path.exists(CVEFIXES_DB):
    # ALWAYS inspect first: this prints the schema your copy actually has.
    !python scripts/dataset/collect_cvefixes.py --db {CVEFIXES_DB} --inspect
    !python scripts/dataset/collect_cvefixes.py --db {CVEFIXES_DB} --limit 200
    print("--- smoke test above. If it produced pairs, rerun without --limit ---")
    !python scripts/dataset/collect_cvefixes.py --db {CVEFIXES_DB}
else:
    print(f"{CVEFIXES_DB} not found - skipping CVEfixes")

## 2. Build the feature matrix

Extracts the 52 features in `FEATURE_NAMES` order and attaches multi-hot labels
in `CLASS_ORDER`. Watch the label-balance table it prints: any class under 30
positives cannot support a per-class F1 you would want to quote.

In [ ]:
!python scripts/build_dataset.py

In [ ]:
import pandas as pd
df = pd.read_csv("data/processed/dataset.csv")
print(df.shape)
print("\nrows per source:"); print(df["source"].value_counts())
print("\nrows per language:"); print(df["language"].value_counts())
print("\ngroups:", df["group"].nunique(), "- the split unit")
df.head(3)

## 3. Train

Group-split, class-weighted BCE, early stopping on validation mean average
precision (threshold-free, which is the right target for a ranker), then
per-class thresholds tuned on validation.

In [ ]:
!python scripts/train_triage.py --data data/processed/dataset.csv --out models/ --epochs 300

## 4. Read the report honestly

`docs/model/test_report.csv` has one row per class. Three things to check before
you quote anything:

* **`status == untrained`** means no positive sample existed. That class is
  *unmeasured*, not zero-performing. Say "not trained" — never imply a zero.
* **`status == too few samples`** means under 30 test positives. An F1 from that
  is noise.
* **`support`** is what any number rests on. Quote it alongside the number.

The model also declines to answer in two cases, and this is deliberate: it
refuses a language absent from its training set, and it refuses a feature vector
that falls outside the range it was trained on. A model trained only on Java has
no basis for an opinion about Python, and emitting 0.02 for it is worse than
emitting nothing.

In [ ]:
import pandas as pd
report = pd.read_csv("docs/model/test_report.csv")
display(report)

ok = report[report["status"] == "ok"]
if len(ok):
    print(f"\nmacro F1 over {len(ok)} classes with >=30 test samples: {ok['f1'].mean():.3f}")
    print("Quote it exactly like that - with the class count and the threshold.")
untrained = report[report["status"] == "untrained"]["class"].tolist()
if untrained:
    print(f"\nUNTRAINED (no positives in this dataset): {untrained}")
    print("These are unmeasured. Do not present them as zero.")

## 5. Is this enough data?

Run this before you decide you are finished. If most classes are untrained, the
honest options are to add a source that covers them or to ship rules-only —
not to quote a macro average over the three classes that happened to have data.

In [ ]:
import json
from codesentinel.triage.model import CLASS_ORDER

label_cols = [f"y_{c}" for c in CLASS_ORDER]
counts = df[label_cols].sum().astype(int)
for cls, n in zip(CLASS_ORDER, counts):
    verdict = "ok" if n >= 100 else ("thin" if n >= 30 else "NOT ENOUGH")
    print(f"  {cls}  {n:6d}  {verdict}")

scaler = json.load(open("models/feature_scaler.json"))
print("\ntrained on languages:", scaler["languages"])
print("Inference will decline every other language. If that list is one item,")
print("the model is useful for that language only - say so.")

## 6. Ship it

Two files, nothing else:

* `models/triage.onnx`
* `models/feature_scaler.json`

Download them from the Kaggle output panel, then either commit them (they are
only a few KB) or attach them to a GitHub release so `cs install-model` can
fetch them.

Then verify on your machine — this is the part that actually matters:

```bash
cs version                 # -> triage model: loaded
cs scan demo/ --no-ledger  # findings unchanged; ordering may differ
pytest -q                  # the model-present tests now run instead of skipping
python scripts/benchmark.py
```

The last one is the honest cost of the model. Run it with `models/` empty and
again with the model present, and quote both.

In [ ]:
from IPython.display import FileLink
import os
for f in ("models/triage.onnx", "models/feature_scaler.json", "docs/model/test_report.csv"):
    if os.path.exists(f):
        print(f, f"{os.path.getsize(f)/1024:.1f} KB")
        display(FileLink(f))